In [0]:
%pip install -r /Workspace/Users/neeraj.koul@dot.gov/GSAHackathon/mcp-fra-safety/requirements.txt fpdf2
dbutils.library.restartPython()

In [0]:
import os
import subprocess
import sys

# 1. Inject active Databricks notebook credentials & SQL Warehouse config
ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()
os.environ["DATABRICKS_HOST"] = ctx.apiUrl().get()
os.environ["DATABRICKS_TOKEN"] = ctx.apiToken().get()
os.environ["DATABRICKS_WAREHOUSE_ID"] = "50d2bd4ddf65e0f5"
os.environ["DATABRICKS_CATALOG"] = "users"
os.environ["DATABRICKS_SCHEMA"] = "neeraj_koul"

project_dir = "/Workspace/Users/neeraj.koul@dot.gov/GSAHackathon/mcp-fra-safety"
os.chdir(project_dir)
sys.path.insert(0, os.path.join(project_dir, "src"))

# 2. Run unit tests
print("=== Running pytest ===")
subprocess.run([sys.executable, "-m", "pytest", "-v"], check=True)

# 3. Run the 19-case Unity Catalog parity benchmark
print("\n=== Running 19-case eval suite ===")
subprocess.run([sys.executable, "eval/run_eval.py"], check=True)

In [0]:
import datetime
import json
import os
from pathlib import Path
import sys
from fpdf import FPDF

# ---------------------------------------------------------------------------
# 1. Bootstrap Environment, Config & Live Data (Self-Contained for Notebook)
# ---------------------------------------------------------------------------
REPO_ROOT = Path("/Workspace/Users/neeraj.koul@dot.gov/GSAHackathon")
PROJECT_ROOT = REPO_ROOT / "mcp-fra-safety"
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

try:
    ctx = dbutils.notebook.entry_point.getDbutils().notebook().getContext()  # type: ignore[name-defined]
    os.environ.setdefault("DATABRICKS_HOST", ctx.apiUrl().get())
    os.environ.setdefault("DATABRICKS_TOKEN", ctx.apiToken().get())
except Exception:
    pass

os.environ.setdefault("DATABRICKS_WAREHOUSE_ID", "50d2bd4ddf65e0f5")
os.environ.setdefault("DATABRICKS_CATALOG", "users")
os.environ.setdefault("DATABRICKS_SCHEMA", "neeraj_koul")

from fra_safety_mcp.config import config  # noqa: E402
from fra_safety_mcp.db import execute_query  # noqa: E402

# Query live Unity Catalog row counts across all 5 FRA + STB datasets
datasets_to_count = [
    ("FRA Form 54 - Rail Equipment Accidents/Incidents", config.form_54_table),
    ("FRA Form 55 - Railroad Operational Exposure", config.form_55_table),
    ("FRA Form 55a - Railroad Injury & Illness Summary", config.form_55a_table),
    ("FRA Form 57 - Highway-Rail Grade Crossing Incidents", config.form_57_table),
    ("STB Part 1246 - Class I Railroad Employment", config.stb_1246_table),
]

table_stats = {}
for label, tbl_name in datasets_to_count:
    fq = config.fq_table(tbl_name)
    try:
        res = execute_query(f"SELECT COUNT(*) AS cnt FROM {fq}")
        table_stats[f"{label} ({tbl_name})"] = int(res[0]["cnt"] or 0)
    except Exception:
        table_stats[f"{label} ({tbl_name})"] = 0

# Load eval/eval_report.json (or run eval/run_eval.py if not generated yet)
report_path = PROJECT_ROOT / "eval" / "eval_report.json"
if not report_path.exists():
    from eval.run_eval import run_benchmark  # noqa: E402
    raw_report = run_benchmark()
else:
    raw_report = json.loads(report_path.read_text(encoding="utf-8"))

case_results = raw_report.get("results", [])
total_cases = int(raw_report.get("total", len(case_results)))
passed_cases = int(raw_report.get("passed", sum(1 for r in case_results if r.get("passed"))))
pass_rate_pct = round((passed_cases / total_cases) * 100, 1) if total_cases else 0.0
latencies = [float(r.get("elapsed_ms", 0.0)) for r in case_results]
avg_latency_ms = round(sum(latencies) / len(latencies), 1) if latencies else 0.0
total_duration_s = round(sum(latencies) / 1000.0, 2)

eval_summary = {
    "tool_cases_passed": passed_cases,
    "tool_cases_total": total_cases,
    "tool_pass_rate_pct": pass_rate_pct,
    "avg_tool_latency_ms": avg_latency_ms,
    "total_duration_s": total_duration_s,
    "resources_total": 6,
    "prompts_total": 5,
    "case_results": [
        {
            "id": r["id"],
            "category": r.get("category", ""),
            "latency_ms": r.get("elapsed_ms", 0.0),
            "passed": bool(r.get("passed", False)),
        }
        for r in case_results
    ],
}

warehouse_id = getattr(config, "warehouse_id", os.environ.get("DATABRICKS_WAREHOUSE_ID", "50d2bd4ddf65e0f5"))


# ---------------------------------------------------------------------------
# 2. Build 3-Page GSA Hackathon Submission PDF
# ---------------------------------------------------------------------------
class HackathonSubmissionPDF(FPDF):
    def header(self):
        self.set_fill_color(15, 44, 89)  # DOT / FRA Navy
        self.rect(0, 0, 210, 18, "F")
        self.set_font("Helvetica", "B", 10)
        self.set_text_color(255, 255, 255)
        self.set_xy(12, 5)
        self.cell(
            0,
            8,
            "GSA HACKATHON | DATASET ACCESS MCP SERVER SUBMISSION - mcp-fra-safety",
            align="L",
        )
        self.ln(16)

    def footer(self):
        self.set_y(-13)
        self.set_font("Helvetica", "I", 8)
        self.set_text_color(100, 110, 125)
        self.cell(
            0,
            8,
            f"U.S. DOT / Federal Railroad Administration (49 CFR Part 225) & STB | Page {self.page_no()}/{{nb}}",
            align="C",
        )

    def section_title(self, title: str):
        self.ln(3)
        self.set_font("Helvetica", "B", 12)
        self.set_text_color(15, 44, 89)
        self.cell(0, 7, title, new_x="LMARGIN", new_y="NEXT")
        self.set_draw_color(30, 100, 180)
        self.set_line_width(0.5)
        self.line(12, self.get_y(), 198, self.get_y())
        self.ln(3)

    def body_text(self, text: str):
        self.set_font("Helvetica", "", 9.5)
        self.set_text_color(35, 40, 50)
        self.multi_cell(0, 5, text)
        self.ln(1.5)

    def bullet_item(self, lead: str, text: str):
        self.set_font("Helvetica", "B", 9.5)
        self.set_text_color(15, 44, 89)
        self.cell(5, 5, "-")
        self.write(5, f"{lead}: ")
        self.set_font("Helvetica", "", 9.5)
        self.set_text_color(35, 40, 50)
        self.write(5, text)
        self.ln(5.5)


pdf = HackathonSubmissionPDF(orientation="P", unit="mm", format="A4")
pdf.alias_nb_pages()
pdf.set_margins(12, 22, 12)
pdf.set_auto_page_break(auto=True, margin=16)

# --- PAGE 1: Executive Summary & Data Foundation ---
pdf.add_page()
pdf.set_font("Helvetica", "B", 17)
pdf.set_text_color(15, 44, 89)
pdf.multi_cell(0, 8, "mcp-fra-safety: 49 CFR Part 225 & STB Rail Safety Dataset Access MCP Server")
pdf.ln(1)

pdf.set_font("Helvetica", "", 10)
pdf.set_text_color(80, 90, 105)
generated_dt = datetime.datetime.now(datetime.timezone.utc).strftime("%Y-%m-%d %H:%M UTC")
pdf.cell(
    0,
    6,
    f"Track: Dataset Access Server | Author: Neeraj Koul (U.S. DOT / FRA) | Generated: {generated_dt}",
    new_x="LMARGIN",
    new_y="NEXT",
)
pdf.ln(3)

# Key Metrics Callout Box
pdf.set_fill_color(240, 245, 252)
pdf.set_draw_color(180, 200, 225)
pdf.rect(12, pdf.get_y(), 186, 20, "DF")
y_box = pdf.get_y() + 3
pdf.set_xy(15, y_box)
pdf.set_font("Helvetica", "B", 9.5)
pdf.set_text_color(15, 44, 89)
total_rows = sum(table_stats.values())
pdf.cell(
    0,
    6,
    f"Governed Records: {total_rows:,} rows (4 FRA Forms + STB Part 1246) | "
    f"Primitives: 9 Tools, 6 Resources, 5 Prompts | "
    f"Eval Score: {eval_summary['tool_cases_passed']}/{eval_summary['tool_cases_total']} "
    f"({eval_summary['tool_pass_rate_pct']}%)",
    new_x="LMARGIN",
    new_y="NEXT",
)
pdf.set_x(15)
pdf.set_font("Helvetica", "", 9)
pdf.set_text_color(50, 60, 75)
pdf.cell(
    0,
    6,
    f"Runtime: Databricks Apps (FastMCP Streamable-HTTP at /mcp) | "
    f"Catalog: {config.catalog}.{config.schema} | Avg Query Latency: {eval_summary['avg_tool_latency_ms']} ms",
)
pdf.set_y(y_box + 20)

pdf.section_title("1. Executive Summary & Cross-Agency Regulatory Problem Solved")
pdf.body_text(
    "Federal Railroad Administration (FRA) safety data under 49 CFR Part 225 spans four interdependent "
    "reporting forms (Forms 54, 55, 55a, and 57), while Class I railroad staffing is reported separately "
    "to the Surface Transportation Board (STB Part 1246). When general-purpose LLMs query raw CSVs or SQL "
    "tables directly, they routinely produce flawed safety metrics: double-counting multi-train collisions "
    "on macro views or under-counting single-carrier involvements when jointcd = '2' or '3' filings are "
    "dropped, mixing highway-rail crossing crashes into Report 2.03 train accident totals, or computing "
    "invalid state-level train-mile rates where Form 55 operational exposure is system-wide."
)
pdf.body_text(
    "mcp-fra-safety solves this by exposing a governed, domain-accurate Model Context Protocol (MCP) server "
    "on Databricks Apps backed by Unity Catalog Delta tables. Every tool enforces official FRA Open Data "
    "portal rules (portals dsuf-xcni, wjun-6kqb, 9hnw-nt86, and filter glossary dnz7-ag8y), bridges FRA "
    "safety outcomes with STB Class I workforce headcount, and pairs read-only SQL analytics with embedded "
    "49 CFR Part 225 methodology resources and multi-step prompts."
)

pdf.section_title("2. Unity Catalog Data Foundation (users.neeraj_koul)")
pdf.set_font("Helvetica", "B", 9)
pdf.set_fill_color(15, 44, 89)
pdf.set_text_color(255, 255, 255)
pdf.cell(146, 7, "  Unity Catalog Governed Dataset", border=1, fill=True)
pdf.cell(40, 7, "Ingested Rows  ", border=1, fill=True, align="R", new_x="LMARGIN", new_y="NEXT")

pdf.set_font("Helvetica", "", 8.8)
pdf.set_text_color(35, 40, 50)
for idx, (lbl, cnt) in enumerate(table_stats.items()):
    fill = idx % 2 == 1
    pdf.set_fill_color(246, 249, 253)
    pdf.cell(146, 6.2, f"  {lbl}", border=1, fill=fill)
    pdf.cell(40, 6.2, f"{cnt:,}  ", border=1, fill=fill, align="R", new_x="LMARGIN", new_y="NEXT")

pdf.section_title("3. Key 49 CFR Part 225 Parity & Governance Guarantees")
pdf.bullet_item(
    "Macro vs. Carrier-Filtered Deduplication",
    "Enforces jointcd = '1' on unfiltered national/state queries to prevent double-counting, while counting "
    "DISTINCT incidentkey across all jointcd filings (1, 2, and 3) when filtered by carrier (achieving 100% "
    "FRA portal parity, e.g., BNSF CY 2021 Report 3.10 = 306).",
)
pdf.bullet_item(
    "Geographic Blank-Rate Rule (Report 1.12)",
    "Automatically sets Form 55 operational exposure and normalized rates (per 1M train-miles or per 200K "
    "employee hours) to None whenever state, county, or city filters are applied, matching FRA portal dsuf-xcni.",
)
pdf.bullet_item(
    "Cross-Agency FRA + STB Workforce Synthesis",
    "Maps AAR reporting marks across FRA Form 54/55/55a and STB Part 1246 employment groups (L100-L600) "
    "to analyze accident and injury rates alongside railroad craft staffing trends.",
)
pdf.bullet_item(
    "Cross-Form Incident Walking & Governed SQL",
    "Walks linked events across Form 54 (incidentkey, gxid), Form 57 (gradecrossingid), and Form 55a "
    "(individual casualties), with strict read-only SQL validation and row-truncation telemetry.",
)

# --- PAGE 2: MCP Architecture & Primitives Catalog ---
pdf.add_page()
pdf.section_title("4. MCP Server Architecture & Primitives (9 Tools, 6 Resources, 5 Prompts)")
pdf.body_text(
    f"Built with FastMCP and deployed on Databricks Apps as a Streamable-HTTP ASGI service (/mcp), "
    f"mcp-fra-safety authenticates to Databricks SQL Warehouse {warehouse_id} via zero-secret OAuth M2M "
    f"service principal credentials and executes parameterized SQL directly against Unity Catalog Delta tables."
)

tools_catalog = [
    ("get_report_1_12_rail_safety_overview", "Forms 54/55a/57/55", "Replicates FRA Report 1.12 overview & normalized rates."),
    ("get_train_accident_summary", "Form 54", "Replicates Reports 2.03, 3.10 & 3.18 (Table 1 macro & Table 2 carrier)."),
    ("audit_joint_track_cause_discrepancies", "Form 54", "Surfaces primary vs. secondary railroad cause & speed mismatches."),
    ("analyze_highway_rail_crossings", "Form 57", "Replicates Reports 1.11, 2.08, 5.14, 5.15 & USDOT crossing hotspots."),
    ("analyze_metro_corridor_safety", "Forms 54/55a/57", "Synthesizes multi-portal rail & crossing risk across metro corridors."),
    ("audit_delta_table_versions", "Unity Catalog", "Tracks Delta DESCRIBE HISTORY & VERSION AS OF retroactive updates."),
    ("walk_across_fra_forms", "Forms 54/57/55a", "Walks linked incidents across accidents, crossings & individual casualties."),
    ("analyze_class1_employment_and_safety", "STB 1246 + FRA", "Joins STB Class I workforce craft counts with FRA safety rates."),
    ("run_fra_sql_query", "All 5 Tables", "Executes read-only cross-agency SQL JOINs with guardrails & alerts."),
]

pdf.set_font("Helvetica", "B", 8.5)
pdf.set_fill_color(15, 44, 89)
pdf.set_text_color(255, 255, 255)
pdf.cell(68, 7, "  MCP Tool Name", border=1, fill=True)
pdf.cell(32, 7, "  Data Scope", border=1, fill=True)
pdf.cell(86, 7, "  Regulatory & Analytical Scope", border=1, fill=True, new_x="LMARGIN", new_y="NEXT")

pdf.set_font("Helvetica", "", 8.2)
pdf.set_text_color(35, 40, 50)
for idx, (t_name, t_forms, t_desc) in enumerate(tools_catalog):
    fill = idx % 2 == 1
    pdf.set_fill_color(246, 249, 253)
    pdf.cell(68, 6.5, f"  {t_name}", border=1, fill=fill)
    pdf.cell(32, 6.5, f"  {t_forms}", border=1, fill=fill)
    pdf.cell(86, 6.5, f"  {t_desc}", border=1, fill=fill, new_x="LMARGIN", new_y="NEXT")

pdf.ln(4)
pdf.bullet_item(
    "6 Methodology Resources (fra://...)",
    "fra://reference/preparing-reports-guide, fra://reports/1.12-methodology, "
    "fra://reports/train-accidents-methodology, fra://reports/form57-crossings-methodology, "
    "fra://reference/filter-glossary, and fra://reference/cause-code-categories.",
)
pdf.bullet_item(
    "5 Workflow Prompts",
    "metro_safety_audit_briefing, railroad_system_safety_scorecard, "
    "railroad_workforce_and_safety_briefing, grade_crossing_hotspot_audit, "
    "and joint_track_discrepancy_review.",
)

# --- PAGE 3: Automated Evaluation Results ---
pdf.add_page()
pdf.section_title("5. Live Unity Catalog Evaluation Benchmark Results")
pdf.body_text(
    f"The automated benchmark suite (eval/run_eval.py) executed {eval_summary['tool_cases_total']} end-to-end "
    f"tool evaluations, {eval_summary['resources_total']} resource reads, and {eval_summary['prompts_total']} "
    f"prompt templates directly against Unity Catalog in {eval_summary['total_duration_s']}s with a "
    f"{eval_summary['tool_pass_rate_pct']}% pass rate."
)

pdf.set_font("Helvetica", "B", 8.5)
pdf.set_fill_color(15, 44, 89)
pdf.set_text_color(255, 255, 255)
pdf.cell(62, 7, "  Benchmark Case ID", border=1, fill=True)
pdf.cell(78, 7, "  Parity Verification Category", border=1, fill=True)
pdf.cell(24, 7, "Latency  ", border=1, fill=True, align="R")
pdf.cell(22, 7, "Status", border=1, fill=True, align="C", new_x="LMARGIN", new_y="NEXT")

pdf.set_font("Helvetica", "", 8)
pdf.set_text_color(35, 40, 50)
for idx, r in enumerate(eval_summary["case_results"]):
    fill = idx % 2 == 1
    pdf.set_fill_color(246, 249, 253)
    status_lbl = "PASS" if r["passed"] else "FAIL"
    pdf.cell(62, 5.8, f"  {r['id']}", border=1, fill=fill)
    pdf.cell(78, 5.8, f"  {r['category']}", border=1, fill=fill)
    pdf.cell(24, 5.8, f"{r['latency_ms']} ms  ", border=1, fill=fill, align="R")
    pdf.cell(22, 5.8, status_lbl, border=1, fill=fill, align="C", new_x="LMARGIN", new_y="NEXT")

out_tmp = Path("/tmp/mcp-fra-safety_GSA_Hackathon_Submission.pdf")
out_ws = REPO_ROOT / "mcp-fra-safety_GSA_Hackathon_Submission.pdf"

pdf.output(str(out_tmp))
try:
    pdf.output(str(out_ws))
    print(f"Saved Submission PDF to Workspace: {out_ws}")
except OSError:
    print(f"Saved Submission PDF to /tmp: {out_tmp}")